In [ ]:
# Some imports

import monai
import torch
from torch.utils.data import DataLoader
from monai.transforms import (
    EnsureChannelFirstd,
    AsDiscreted,
    Compose,
    LoadImaged,
    Orientationd,
    Randomizable,
    Resized,
    ScaleIntensityd,
    Spacingd,
    EnsureTyped,
    Lambda
)
import monai.metrics as metrics
import os
import tempfile
from utils.decathlon_dataset import get_decathlon_dataloader
from utils.unet import UNET
from utils.train import train

import matplotlib.pyplot as plt

In [ ]:
root_dir = './utils/datasets'
task = "Task04_Hippocampus"

train_loader = get_decathlon_dataloader(root_dir, task, "training", batch_size=4, num_workers=0, shuffle=True)
val_loader = get_decathlon_dataloader(root_dir, task, "validation", batch_size=4, num_workers=0, shuffle=False)

d:\New folder\AI\U_net_practice\.venv\Lib\site-packages\monai\utils\deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


2026-11-20 00:50:58,261 - INFO - Verified 'Task04_Hippocampus.tar', md5: 9d24dba78a72977dbd1d2e110310f31b.
2026-11-20 00:50:58,261 - INFO - File exists: utils\datasets\Task04_Hippocampus.tar, skipped downloading.
2026-11-20 00:50:58,262 - INFO - Non-empty folder exists in utils\datasets\Task04_Hippocampus, skipped extracting.


Loading dataset: 100%|██████████| 208/208 [00:01<00:00, 118.96it/s]


2026-11-20 00:51:01,005 - INFO - Verified 'Task04_Hippocampus.tar', md5: 9d24dba78a72977dbd1d2e110310f31b.
2026-11-20 00:51:01,005 - INFO - File exists: utils\datasets\Task04_Hippocampus.tar, skipped downloading.
2026-11-20 00:51:01,006 - INFO - Non-empty folder exists in utils\datasets\Task04_Hippocampus, skipped extracting.


Loading dataset: 100%|██████████| 52/52 [00:00<00:00, 117.95it/s]


In [ ]:
# Load UNET model from MONAI

model = monai.networks.nets.UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=3,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
)

# Define loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Define device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
from utils.train import train
# Your code here
trained_model = None

trained_model = train(model, train_loader, loss_fn, optimizer, device, epochs=20)

Epoch 1/20, Loss: 1.0579
Epoch 2/20, Loss: 0.5301
Epoch 3/20, Loss: 0.2444
Epoch 4/20, Loss: 0.1556
Epoch 5/20, Loss: 0.1255
Epoch 6/20, Loss: 0.1105
Epoch 7/20, Loss: 0.1018
Epoch 8/20, Loss: 0.0943
Epoch 9/20, Loss: 0.0876
Epoch 10/20, Loss: 0.0811
Epoch 11/20, Loss: 0.0747
Epoch 12/20, Loss: 0.0679
Epoch 13/20, Loss: 0.0620
Epoch 14/20, Loss: 0.0569
Epoch 15/20, Loss: 0.0534
Epoch 16/20, Loss: 0.0495
Epoch 17/20, Loss: 0.0463
Epoch 18/20, Loss: 0.0439
Epoch 19/20, Loss: 0.0414
Epoch 20/20, Loss: 0.0392


In [ ]:
import torch
from monai.metrics import HausdorffDistanceMetric

hausdorff_distance = HausdorffDistanceMetric(include_background=False, reduction="mean", percentile=95)

for val_data in val_loader:
    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
    val_outputs = torch.nn.functional.one_hot(trained_model(val_inputs).argmax(dim=1), num_classes=3).permute(0, 3, 1, 2)
    val_labels = torch.nn.functional.one_hot(val_labels.long(), num_classes=3).squeeze(1).permute(0, 3, 1, 2)

    # Identify valid classes
    valid_classes = torch.logical_and(torch.any(val_labels, dim=(0, 2, 3)), torch.any(val_outputs, dim=(0, 2, 3)))

    # Only compute Hausdorff distance for valid classes
    if torch.any(valid_classes):
        hausdorff_distance(val_outputs[:, valid_classes], val_labels[:, valid_classes])
    else:
        print("No valid classes found for Hausdorff distance calculation.")

result = hausdorff_distance.aggregate().item()
print(f'95th percentile Hausdorff distance: {result:.4f}')

d:\New folder\AI\U_net_practice\.venv\Lib\site-packages\monai\utils\deprecate_utils.py:220: FutureWarning: monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
  warn_deprecated(argname, msg, warning_category)
d:\New folder\AI\U_net_practice\.venv\Lib\site-packages\monai\metrics\hausdorff_distance.py:214: UserWarning: the ground truth of class 1 is all 0, this may result in nan/inf distance.
  _, distances, _ = get_edge_surface_distance(
d:\New folder\AI\U_net_practice\.venv\Lib\site-packages\monai\metrics\hausdorff_distance.py:214: UserWarning: the prediction of class 1 is all 0, this may result in nan/inf distance.
  _, distances, _ = get_edge_surface_distance(
d:\New folder\AI\U_net_practice\.venv\Lib\site-packages\monai\metrics\hausdorff_distance.py:214: UserWarning: the ground truth o

95th percentile Hausdorff distance: 2.5303
